### TrOCR: test set evaluation only

Pretrained `microsoft/trocr-base-handwritten` on **Test_Label.csv** and **Test_Set** images. Reports corpus **WER**, **CER**, and exact line accuracy. **CUDA required** (no CPU fallback).
**WSL2:** the first code cell prepends `/usr/lib/wsl/lib` to `LD_LIBRARY_PATH` so PyTorch can load `libcuda`. Run cells top-to-bottom after a **kernel restart**.

In [1]:
import os
from pathlib import Path

# WSL2: without this, torch often loads with cuda built-in (+cu12x) but cuda.is_available() is False
_wsl = Path("/usr/lib/wsl/lib")
if _wsl.is_dir():
    os.environ["LD_LIBRARY_PATH"] = (
        f"{_wsl}{os.pathsep}{os.environ.get('LD_LIBRARY_PATH', '')}"
    ).strip(os.pathsep)

import torch
import numpy as np
import pandas as pd
from jiwer import cer, wer
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DATA_ROOT = Path("real_dataset/RxHandBD-ML")
TEST_CSV = DATA_ROOT / "Test_Label.csv"
PRETRAINED = "microsoft/trocr-base-handwritten"
GEN_MAX_LEN = 64
GEN_NUM_BEAMS = 1
EVAL_BATCH = 4

if not torch.cuda.is_available():
    import sys
    n = torch.cuda.device_count()
    raise RuntimeError(
        f"TrOCR eval needs CUDA, but torch.cuda.is_available() is False.\n"
        f"  python: {sys.executable!r}\n"
        f"  torch:  {torch.__version__!r}  built_for_cuda: {torch.version.cuda!r}\n"
        f"  WSL2:  ensure /usr/lib/wsl/lib exists, run: nvidia-smi\n"
        f"  If nvidia-smi works in a shell but this fails, restart the kernel and run this cell first.\n"
        f"  torch.cuda.device_count()={n}"
    )

device = torch.device("cuda")
test_df = pd.read_csv(TEST_CSV)
test_df.columns = [c.strip() for c in test_df.columns]


In [2]:
def align_model_to_processor(model, processor) -> None:
    tok = processor.tokenizer
    p = getattr(model.config, "pad_token_id", None)
    if p is None and tok.pad_token_id is not None:
        model.config.pad_token_id = int(tok.pad_token_id)
    d = getattr(model.config, "decoder_start_token_id", None)
    if d is None and getattr(model.config, "pad_token_id", None) is not None:
        model.config.decoder_start_token_id = int(model.config.pad_token_id)


def corpus_metrics(refs: list[str], hyps: list[str]) -> dict[str, float]:
    r = [str(x) for x in refs]
    h = [str(x) for x in hyps]
    return {
        "wer": float(wer(r, h)),
        "cer": float(cer(r, h)),
        "seq_acc": float(np.mean([a.strip() == b.strip() for a, b in zip(r, h)])),
    }


processor = TrOCRProcessor.from_pretrained(PRETRAINED, backend="pil")
model = VisionEncoderDecoderModel.from_pretrained(PRETRAINED).to(device)
align_model_to_processor(model, processor)
model.eval()

pad_id = int(processor.tokenizer.pad_token_id)
eos_tok = processor.tokenizer.eos_token_id
eos_id = int(eos_tok if eos_tok is not None else pad_id)

names = test_df["Images"].astype(str).tolist()
refs = test_df["Text"].astype(str).tolist()
paths = [DATA_ROOT / "Test_Set" / n for n in names]
preds: list[str] = []

for s0 in range(0, len(paths), EVAL_BATCH):
    part = paths[s0 : s0 + EVAL_BATCH]
    imgs = [Image.open(p).convert("RGB") for p in part]
    enc = processor(images=imgs, return_tensors="pt", padding=True)
    px = enc["pixel_values"].to(device)
    gen = model.generate(
        px,
        max_length=GEN_MAX_LEN,
        num_beams=GEN_NUM_BEAMS,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
    )
    preds.extend(
        processor.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    )

m = corpus_metrics(refs, preds)
print("n_samples:", len(refs))
print("wer:", m["wer"])
print("cer:", m["cer"])
print("line_acc (exact):", m["seq_acc"])


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


n_samples: 1115
wer: 1.4575586095392077
cer: 0.7053924914675768
line_acc (exact): 0.07892376681614349


### Fine-Tuning TrOCR on RxHandBD